In [ ]:
# Imports
import numpy as np
import pandas as pd
import gc

# Reusable framework code (installed via `pip install -e .` from the repo root)
from soyini import config
from soyini.seasonal import select_seasonal_data
from soyini.indices import compute_swei
from soyini.plotting import plot_index_heatmap

In [ ]:
# Configuration (paths resolved by soyini.config; set SD_DATA_ROOT or
# config/paths.yaml to point at your data tree)
casr_input_dir = config.output_data("CaSR", ensure=False)          # input: combined_precip_swe_*.csv
shapefile = config.elevation_shapefile()
output_dir_SWEI = config.output_data("SWEI")
output_plots = config.output_plots("SWEI")

# Select data for winter and combine data for SWEI calculation

In [ ]:
files = sorted(casr_input_dir.glob("combined_precip_swe_*.csv"))

dfs = []

for file in files:
    year = int(file.stem.split("_")[-1])
    print("Reading", year)

    df = pd.read_csv(
        file,
        usecols=["Grid_ID", "time", "lat", "lon", "elev_class", "swe", "precip"]
    )

    df["time"] = pd.to_datetime(df["time"])

    # keep winter months only: Oct-May
    df = df[df["time"].dt.month.isin([10, 11, 12, 1, 2, 3, 4, 5])]

    # seasonal year: Oct-Dec belong to same year, Jan-May previous year
    df['Seasonal_Year'] = df['time'].apply(
        lambda ts: select_seasonal_data(ts, start_month=10, end_month=5, min_year=1980, max_year=2023)
    )


    dfs.append(df)

daily_all = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(daily_all.shape)

In [ ]:
# drop NaN in Seasonal_Year
daily_all = daily_all.dropna(subset=["Seasonal_Year"])

#rename swe to SWE
daily_all = daily_all.rename(columns={"swe": "SWE"})

# rename precip to Precipitation
daily_all = daily_all.rename(columns={"precip": "Precipitation"})

#remove decimal in Seasonal_Year
daily_all["Seasonal_Year"] = daily_all["Seasonal_Year"].astype(int)

# save daily_all to csv
daily_all.to_csv(output_dir_SWEI / 'Alberta_casr_daily_all_new.csv', index=False)
display(daily_all.head())

# Calculate SWEI

In [ ]:
#8-month SWEI
swei_8mo = compute_swei(daily_all, window_months=8, seed=42)
swei_8mo.to_csv(output_dir_SWEI / 'bow_casr_swei_8mo_new.csv', index=False)

display(swei_8mo.head())

In [17]:
# avaerage SWEI over the Elevation categories, month and seasonal year
swei_8mo_avg = (
    swei_8mo
    .groupby(
        ['elev_class', 'Seasonal_Year','month'],
        as_index=False
    )
    .agg(
        Avg_SWEI_8mo=('SWEI', 'mean')
    )
)

display(swei_8mo_avg.head(15))

,elev_class,Seasonal_Year,month,Avg_SWEI_8mo
0,0_500m,1980,5,-0.319014
1,0_500m,1981,5,0.249791
2,0_500m,1982,5,0.710513
3,0_500m,1983,5,-0.909618
4,0_500m,1984,5,0.544012
5,0_500m,1985,5,0.442175
6,0_500m,1986,5,0.983767
7,0_500m,1987,5,0.022860
8,0_500m,1988,5,1.342846
9,0_500m,1989,5,1.331155


In [ ]:
# save swei_8mo_avg to csv
swei_8mo_avg.to_csv(output_dir_SWEI / 'Alberta_casr_swei_8mo_avg_new.csv', index=False)

In [ ]:
#print snow drought years for each elevation category
for elev_cat in swei_8mo_avg['elev_class'].unique():
    elev_data = swei_8mo_avg[swei_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SWEI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    print(f"Elevation Category: {elev_cat}, Drought Years (Avg SWEI <= -0.5): {drought_years.tolist()}")

# Create a dataframe with drought years for each elevation category
drought_data = []
for elev_cat in swei_8mo_avg['elev_class'].unique():
    elev_data = swei_8mo_avg[swei_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SWEI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    drought_data.append({
        'elev_class': elev_cat,
        'Drought_Years': sorted(drought_years.tolist())
    })

drought_years_df = pd.DataFrame(drought_data)

# save the drought years dataframe to a CSV file
drought_years_df.to_csv(output_dir_SWEI / "Drought_Years_by_Elevation.csv", index=False)  

display(drought_years_df)

In [ ]:
# Seasonal SWEI heatmap (elevation x seasonal year).
# Row order + styling live in soyini.plotting.plot_index_heatmap.
plot_index_heatmap(
    swei_8mo_avg,
    value_col="Avg_SWEI_8mo",
    title="Seasonal SWEI (Oct–May)",
    out_file=output_plots / "SWEI_Alberta_basin_8month_heatmap.png",
)